# AI Agent Demo (LangGraph + MCP + skills)

This notebook drives the asyncroscopy MCP server with the LangGraph agent layer in `asyncroscopy.agent`:

- **Model choice**: run the graphs against a local **Ollama** model, an **API-key provider** (OpenAI, Anthropic, ...), or the **LLM Tango device** running on another computer. Pick one in the *Choose a model* section below.
- **ReAct agent + skills**: a single agent with all MCP tools plus the `skills/` library (Hermes-style `SKILL.md` files) for less deterministic tasks.
- **Deterministic workflow**: `image_eds_survey`, a fixed LangGraph graph (acquire HAADF image -> acquire EDS spectrum -> read back -> summarise) for reproducible runs.
- **LangGraph Studio**: the same graphs are exposed through `langgraph.json` (`uv run langgraph dev`).

LangGraph is where deterministic, reviewable workflows live; skills give the agent context for open-ended requests.

## Prerequisites

Start the microscope stack (the DigitalTwin works without hardware) and the MCP server:

```bash
uv run startup_scripts/run_servers.py --yaml configs/DigitalTwin.yaml
uv run startup_scripts/run_mcp.py --yaml configs/mcp_dt.yaml
```

Install the agent extras (add `--extra ollama` for local models, `--extra studio` for LangGraph Studio):

```bash
uv sync --extra agent --extra ollama --extra studio
cp .env.example .env    # optional: provider, model, API keys
```

Only for the *LLM Tango device* option, also start the device (on the LLM computer):

```bash
uv run startup_scripts/run_llm.py --yaml configs/gemma-llm.yaml
```

## Imports and connections

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tango
from IPython.display import Image, display
from langchain_core.messages import HumanMessage
from tiled.client import from_uri

from asyncroscopy.agent.config import AgentSettings, resolve_api_key
from asyncroscopy.agent.graphs.react import build_react_graph
from asyncroscopy.agent.graphs.workflows.image_eds_survey import build_image_eds_survey_graph
from asyncroscopy.agent.models import TangoChatModel, build_chat_model
from asyncroscopy.agent.skills import SkillRegistry, render_skills_index
from asyncroscopy.agent.streaming import stream_agent
from asyncroscopy.agent.tools import load_mcp_tools

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent

MICROSCOPE_HOST = "localhost"   # Tango database host (microscope computer)
DB_PORT = 9094
MCP_URL = "http://127.0.0.1:8000/mcp"

os.environ["TANGO_HOST"] = f"{MICROSCOPE_HOST}:{DB_PORT}"
print("TANGO_HOST =", os.environ["TANGO_HOST"])

In [ ]:
microscope = tango.DeviceProxy("asyncroscopy/instrument/default")
data = tango.DeviceProxy("asyncroscopy/data/default")
for proxy in (microscope, data):
    proxy.set_timeout_millis(120_000)
    proxy.ping()
    print(proxy.name(), proxy.state())

tiled_config = json.loads(data.get_config())
client = from_uri(tiled_config["uri"])
print("Tiled:", tiled_config["uri"])

## Choose a model

Run **exactly one** of the next three cells. Each one defines `model`, a LangChain chat model that every graph below uses.

| Option | Where inference runs | Needs |
|--------|----------------------|-------|
| Ollama | this computer | `ollama` installed, a tool-calling model pulled (`ollama pull gemma4:31b`, `qwen3:8b`, `llama3.1`, ...) |
| API key | provider cloud | `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` in your environment or `.env` |
| LLM Tango device | the LLM computer | `run_llm.py` running; only Tango access from here |

### Option 1: Ollama (local)

In [ ]:
settings = AgentSettings(provider="ollama", model="gemma4:31b")   # or "qwen3:8b", "llama3.1"
# settings.base_url = "http://other-machine:11434"                 # if Ollama runs elsewhere
model = build_chat_model(settings)
print(model.invoke("Reply with the single word: ready").content)

### Option 2: API key (OpenAI, Anthropic, Google, ...)

The key is read from the environment variable named by `api_key_env` (or the provider's usual variable). It is never written to disk by this notebook; if it is missing you are prompted for it.

In [ ]:
from getpass import getpass

settings = AgentSettings(provider="openai", model="gpt-4o-mini", api_key_env="OPENAI_API_KEY")
# settings = AgentSettings(provider="anthropic", model="claude-sonnet-4-5", api_key_env="ANTHROPIC_API_KEY")
# settings = AgentSettings(provider="google_genai", model="gemini-2.5-flash", api_key_env="GOOGLE_API_KEY")

if resolve_api_key(settings) is None:
    os.environ[settings.api_key_env] = getpass(f"Enter {settings.api_key_env}: ")

model = build_chat_model(settings)
print(model.invoke("Reply with the single word: ready").content)

### Option 3: LLM Tango device (current setup)

Inference happens on the LLM computer through the device's `Complete` command; the graphs still run here. The device is configured with `configs/gemma-llm.yaml` (Ollama) or `configs/openai-llm.yaml` (API key).

In [ ]:
llm = tango.DeviceProxy("asyncroscopy/llm/default")
llm.set_timeout_millis(300_000)
print(llm.name(), llm.state())

model = TangoChatModel(device_name="asyncroscopy/llm/default", timeout_ms=300_000)
print(model.invoke("Reply with the single word: ready").content)

## Load MCP tools and skills

MCP tools are named `<DeviceClass>_<command>` (plus `list_devices` and `get_data_from_key`). Skills are `SKILL.md` files under `skills/`; only their names and descriptions go into the system prompt, the agent loads a body with `skill_view` when it needs it.

In [ ]:
tools = await load_mcp_tools(MCP_URL)
print(f"{len(tools)} MCP tools:")
for t in tools:
    print(" -", t.name)

registry = SkillRegistry.from_dirs([REPO / "skills"])
print()
print(render_skills_index(registry))

## ReAct agent with skills

In [ ]:
agent = build_react_graph(model, tools, registry)
print(agent.get_graph().draw_mermaid())
try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as exc:  # rendering needs network access to mermaid.ink
    print("(PNG rendering skipped:", exc, ")")

In [ ]:
prompt = "Acquire a HAADF image, then an EDS spectrum, and tell me the composition."
answer = await stream_agent(agent, [HumanMessage(content=prompt)], label="react")
print(answer)

## Deterministic workflow: image then EDS survey

`build_image_eds_survey_graph` wires fixed nodes to fixed MCP tools; the model (optional) only rephrases the final summary. Pass `model=None` for a fully deterministic run.

In [ ]:
survey = build_image_eds_survey_graph(tools, model=model)
print(survey.get_graph().draw_mermaid())

result = await survey.ainvoke({"detector": "haadf", "spectrum_detector": "eds"})
print(result["summary"])
print("image key   :", result.get("image_key"))
print("spectrum key:", result.get("spectrum_key"))
if result.get("errors"):
    print("errors      :", result["errors"])

### Plot the HAADF image

In [ ]:
image_key = result["image_key"]
image = client[image_key]["image"]["HAADF"].read()

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap="gray", interpolation="none")
ax.set_title(image_key)
ax.axis("off")
plt.tight_layout()

### Plot the EDS spectrum

In [ ]:
spectrum_key = result["spectrum_key"]
node = client[spectrum_key]["spectrum"]
spectrum = np.squeeze(np.asarray(node.read()))
metadata = dict(node.metadata)
elements = metadata.get("elements")
if isinstance(elements, str):
    elements = json.loads(elements)

plt.figure(figsize=(6, 4))
if elements and len(elements) == spectrum.size:
    plt.bar(elements, spectrum)
    plt.ylabel("relative intensity")
else:
    plt.plot(spectrum)
    plt.xlabel("channel")
    plt.ylabel("counts")
plt.title(spectrum_key)
plt.show()

## Let the agent write a skill

The `skill_save` tool records a procedure as `skills/<name>/SKILL.md` so future runs (and other people) can reuse it. Review it like any other file before committing.

In [ ]:
prompt = (
    "Save a skill named 'quick-survey' that describes how to acquire a HAADF image and an EDS spectrum "
    "and report the composition, naming the exact tools and arguments you would use."
)
answer = await stream_agent(agent, [HumanMessage(content=prompt)], label="react")
print(answer)

registry.reload()
skill = registry.get("quick-survey")
if skill:
    print("Saved to:", skill.path)
    print(skill.body[:500])

## Everything on the LLM Tango device

The device also runs the same graphs on its own: `Query` runs the (skills-aware) worker agents, and `RunWorkflow` runs a deterministic workflow by name. Requires `run_llm.py` to be running.

In [ ]:
llm = tango.DeviceProxy("asyncroscopy/llm/default")
llm.set_timeout_millis(300_000)

print("agents :", list(llm.agents))
print("skills :", [s["name"] for s in json.loads(llm.skills)])

# Add a worker restricted to spectrum tools (tool entries are glob patterns).
llm.SpawnAgent(json.dumps({
    "name": "eds",
    "system_prompt": "You are a worker agent responsible for acquiring EDS spectra.",
    "tools": ["*_acquire_spectrum", "get_data_from_key"],
    "description": "Acquires EDS spectra using the available tools.",
}))

# adjust llm.max_steps if the LLM cannot complete a task
print(llm.Query("Get a scanned haadf image. Then get an EDS spectrum."))

workflow_result = json.loads(llm.RunWorkflow(json.dumps({"name": "image_eds_survey", "input": {"detector": "haadf"}})))
print(workflow_result.get("summary") or workflow_result)

## Next: LangGraph Studio

The three graphs (`react_agent`, `supervisor`, `image_eds_survey`) are registered in `langgraph.json`. With the stack running:

```bash
cp .env.example .env   # set ASYNCROSCOPY_AGENT_PROVIDER / MODEL and any API key
uv run langgraph dev
```

Studio (opened in your browser) visualises each graph, lets you run it with different inputs and `configurable` values (`provider`, `model`, `mcp_url`, ...), and inspect or replay every node's state. New nodes and workflows are written in Python under `asyncroscopy/agent/graphs/` and appear in Studio after a reload; see `docs/Agent/langgraph_studio.md`.